In [1]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime
headers = {"User-Agent": "MyDataResearchProject-dev-1.0.0", "Accept": "application/ld+json"}

In [94]:
df = pd.read_csv("all_thursdays.csv")

In [3]:
df.head()

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,0,eli/dl/event/MTG-PL-2024-01-18,Activity,2024-01-18,2024-01-18T23:00:00+01:00,MTG-PL-2024-01-18,"{'lt': 'Ketvirtadienis, 2024 m. sausio 18 d.',...",2024-01-18T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-93993...,['eli/dl/doc/OJQ-9-2024-01-18'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197478', 'person/132...","['person/4746', 'person/197772', 'person/24457...",586.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-18-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
1,1,eli/dl/event/MTG-PL-2024-01-25,Activity,2024-01-25,2024-01-25T23:00:00+01:00,MTG-PL-2024-01-25,"{'de': 'Donnerstag, 25. Januar 2024', 'it': 'G...",2024-01-25T01:00:00+01:00,"['eli/dl/event/MTG-PL-2024-01-25-PVCRE-ITM-5',...",['eli/dl/doc/OJQ-9-2024-01-25'],def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['eli/dl/doc/CRE-9-2024-01-25', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN
2,2,eli/dl/event/MTG-PL-2024-02-08,Activity,2024-02-08,2024-02-08T23:00:00+01:00,MTG-PL-2024-02-08,"{'sl': 'Četrtek, 8. februar 2024', 'pl': 'Czwa...",2024-02-08T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-02-08-PVCRE-ITM-11'...,['eli/dl/doc/OJQ-9-2024-02-08'],def/ep-activities/PLENARY_SITTING,"['person/250572', 'person/197671', 'person/112...","['person/197563', 'person/185341', 'person/197...",578.0,org/ep-9,"['eli/dl/doc/PV-9-2024-02-08-RCV', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
3,3,eli/dl/event/MTG-PL-2024-02-29,Activity,2024-02-29,2024-02-29T23:00:00+01:00,MTG-PL-2024-02-29,"{'et': 'Neljapäev, 29. veebruar 2024', 'da': '...",2024-02-29T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-02-29-PVCRE-ITM-14'...,['eli/dl/doc/OJQ-9-2024-02-29'],def/ep-activities/PLENARY_SITTING,"['person/197796', 'person/124770', 'person/250...","['person/197590', 'person/236050', 'person/197...",567.0,org/ep-9,"['eli/dl/doc/PV-9-2024-02-29-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
4,4,eli/dl/event/MTG-PL-2024-03-14,Activity,2024-03-14,2024-03-14T23:00:00+01:00,MTG-PL-2024-03-14,"{'fr': 'Jeudi 14 mars 2024', 'et': 'Neljapäev,...",2024-03-14T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-03-14-PVCRE-ITM-13'...,['eli/dl/doc/OJQ-9-2024-03-14'],def/ep-activities/PLENARY_SITTING,"['person/197457', 'person/96697', 'person/1975...","['person/197584', 'person/128483', 'person/197...",566.0,org/ep-9,"['eli/dl/doc/CRE-9-2024-03-14', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN


In [4]:
#cleaning
df["id"] = df["id"].str.replace("eli/dl/event/", "")
df.drop(columns=["activity_end_date", "activity_id", "activity_label", "activity_start_date",], inplace=True)
df["meeting_minutes"] = df["documented_by_a_realization_of"]
df.drop(columns=["documented_by_a_realization_of"], inplace=True)
df["recorded_in_a_realization_of"] = df["recorded_in_a_realization_of"].str.replace("eli/dl/doc/", "")

,Unnamed: 0,id,type,activity_date,consists_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in,meeting_minutes
0,0,MTG-PL-2024-01-18,Activity,2024-01-18,['eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-93993...,def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197478', 'person/132...","['person/4746', 'person/197772', 'person/24457...",586.0,org/ep-9,"['PV-9-2024-01-18-ATT', 'PV-9-2024-01-18', 'PV...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-01-18']
1,1,MTG-PL-2024-01-25,Activity,2024-01-25,"['eli/dl/event/MTG-PL-2024-01-25-PVCRE-ITM-5',...",def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['CRE-9-2024-01-25', 'PV-9-2024-01-25-ATT', 'P...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-01-25']
2,2,MTG-PL-2024-02-08,Activity,2024-02-08,['eli/dl/event/MTG-PL-2024-02-08-PVCRE-ITM-11'...,def/ep-activities/PLENARY_SITTING,"['person/250572', 'person/197671', 'person/112...","['person/197563', 'person/185341', 'person/197...",578.0,org/ep-9,"['PV-9-2024-02-08-RCV', 'PV-9-2024-02-08-VOT',...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-02-08']
3,3,MTG-PL-2024-02-29,Activity,2024-02-29,['eli/dl/event/MTG-PL-2024-02-29-PVCRE-ITM-14'...,def/ep-activities/PLENARY_SITTING,"['person/197796', 'person/124770', 'person/250...","['person/197590', 'person/236050', 'person/197...",567.0,org/ep-9,"['PV-9-2024-02-29-ATT', 'PV-9-2024-02-29-VOT',...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-02-29']
4,4,MTG-PL-2024-03-14,Activity,2024-03-14,['eli/dl/event/MTG-PL-2024-03-14-PVCRE-ITM-13'...,def/ep-activities/PLENARY_SITTING,"['person/197457', 'person/96697', 'person/1975...","['person/197584', 'person/128483', 'person/197...",566.0,org/ep-9,"['CRE-9-2024-03-14', 'PV-9-2024-03-14', 'PV-9-...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-03-14']


In [5]:
df["consists_of"] = df["consists_of"].str.replace("eli/dl/event/", "")

In [6]:
df.head()

,Unnamed: 0,id,type,activity_date,consists_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in,meeting_minutes
0,0,MTG-PL-2024-01-18,Activity,2024-01-18,"['MTG-PL-2024-01-18-VOT-ITM-939931', 'MTG-PL-2...",def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197478', 'person/132...","['person/4746', 'person/197772', 'person/24457...",586.0,org/ep-9,"['PV-9-2024-01-18-ATT', 'PV-9-2024-01-18', 'PV...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-01-18']
1,1,MTG-PL-2024-01-25,Activity,2024-01-25,"['MTG-PL-2024-01-25-PVCRE-ITM-5', 'MTG-PL-2024...",def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['CRE-9-2024-01-25', 'PV-9-2024-01-25-ATT', 'P...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-01-25']
2,2,MTG-PL-2024-02-08,Activity,2024-02-08,"['MTG-PL-2024-02-08-PVCRE-ITM-11', 'MTG-PL-202...",def/ep-activities/PLENARY_SITTING,"['person/250572', 'person/197671', 'person/112...","['person/197563', 'person/185341', 'person/197...",578.0,org/ep-9,"['PV-9-2024-02-08-RCV', 'PV-9-2024-02-08-VOT',...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-02-08']
3,3,MTG-PL-2024-02-29,Activity,2024-02-29,"['MTG-PL-2024-02-29-PVCRE-ITM-14', 'MTG-PL-202...",def/ep-activities/PLENARY_SITTING,"['person/197796', 'person/124770', 'person/250...","['person/197590', 'person/236050', 'person/197...",567.0,org/ep-9,"['PV-9-2024-02-29-ATT', 'PV-9-2024-02-29-VOT',...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-02-29']
4,4,MTG-PL-2024-03-14,Activity,2024-03-14,"['MTG-PL-2024-03-14-PVCRE-ITM-13', 'MTG-PL-202...",def/ep-activities/PLENARY_SITTING,"['person/197457', 'person/96697', 'person/1975...","['person/197584', 'person/128483', 'person/197...",566.0,org/ep-9,"['CRE-9-2024-03-14', 'PV-9-2024-03-14', 'PV-9-...",http://publications.europa.eu/resource/authori...,NaN,['eli/dl/doc/OJQ-9-2024-03-14']


In [96]:

import ast
first_row_items = df["consists_of"].iloc[0]

if isinstance(first_row_items, str):
    first_row_items = ast.literal_eval(first_row_items)
    
for event_id in first_row_items:
        
        if "VOT" in event_id:
            print(event_id)

eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939931
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939943
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939969
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939933
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939803
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939945
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939926
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939976
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939907
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939935
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939934
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939975
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939974
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939965
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939919
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939917
eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939967


In [60]:
test_id = "MTG-PL-2024-01-18"
url = f"https://data.europarl.europa.eu/api/v2/meetings/{test_id}/vote-results"
response = requests.get(url, headers=headers)
print(response.status_code)        

200


In [61]:
test_data = response.json()
print(test_data.keys())

dict_keys(['data', '@context'])


In [119]:
test_data["data"][11]["inverse_consists_of"][1]

{'id': 'eli/dl/proc/2023-2866',
 'consists_of': ['eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-939965']}

In [84]:
#Each VOT event contains a bunch of DEC events, which have the data we want!
#Path to the list of DEC within VOT-event
print(test_data["data"][0]["consists_of"])

['eli/dl/event/MTG-PL-2024-01-18-DEC-163069', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163487', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163070', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163063', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163490', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163068', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163486', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163484', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163064', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163062', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163488', 'eli/dl/event/MTG-PL-2024-01-18-DEC-163491']


In [66]:
test_2 = "MTG-PL-2024-01-18"
url = f"https://data.europarl.europa.eu/api/v2/meetings/{test_2}/decisions"
response = requests.get(url, headers=headers)
print(response.status_code)

401


In [72]:
# We use the FULL raw ID exactly as it appeared in your 'consists_of' list
decision_id = "eli/dl/event/MTG-PL-2024-01-18-DEC-163070"

# Notice there is NO "/api/v2/" in this URL. We go straight to the root domain.
url = f"https://data.europarl.europa.eu/{decision_id}"

headers = {
    "Accept": "application/ld+json",
    "User-Agent": "MyDataResearchProject-dev-1.0.0"
}

response = requests.get(url, headers=headers)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("Success! We bypassed the broken API gateway.\n")
    
    # Parse the data
    decision_data = response.json()
    
    # Print it out nicely
    print(json.dumps(decision_data, indent=4))
else:
    print(f"Failed with status {response.status_code}: {response.text}")

Status Code: 200
Success! We bypassed the broken API gateway.

{
    "data": [
        {
            "id": "eli/dl/event/MTG-PL-2024-01-18-DEC-163070",
            "type": "Vote",
            "activity_date": "2024-01-18",
            "activity_id": "MTG-PL-2024-01-18-DEC-163070",
            "activity_label": {
                "mt": "A9-0434/2023 - Maria Grapini - Mozzjoni g\u0127al ri\u017coluzzjoni (it-test kollu)",
                "sl": "A9-0434/2023 \u2013 Maria Grapini \u2013 predlog resolucije (celotno besedilo)",
                "hu": "A9-0434/2023 \u2013 Maria Grapini \u2013 \u00c1ll\u00e1sfogl\u00e1si ind\u00edtv\u00e1ny (a sz\u00f6veg eg\u00e9sze)",
                "sv": "A9-0434/2023 - Maria Grapini - Resolutionsf\u00f6rslag (texten i sin helhet)",
                "de": "A9-0434/2023 - Maria Grapini - Entschlie\u00dfungsantrag (gesamter Text)",
                "et": "A9-0434/2023 - Maria Grapini - Resolutsiooni ettepanek (terviktekst)",
                "sk": "A9-0434/2023 -

In [77]:
#path to number of attendees
print(decision_data["data"][0]["number_of_attendees"])
#path to in favor
print(decision_data["data"][0]["number_of_votes_against"])
#path to against
print(decision_data["data"][0]["number_of_votes_favor"])

560
48
446


In [80]:
#path to decision outcome
print(decision_data["data"][0]["decision_outcome"])
#path to type of vote
print(decision_data["data"][0]["decision_method"])
#path to start time!!!
print(decision_data["data"][0]["activity_start_date"])

def/ep-statuses/ADOPTED
def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL
2024-01-18T12:17:04+01:00


In [71]:
#Okay so the codes with DEC record actual voting outcomes
#under the key "decision_outcome"
#path is: decision_data["data"][0]["decision_outcome"]

In [97]:

# Loop through the massive JSON list the server gave you
#Actually we're gonna loop through it and make the api calls in a separate notebook
items_list = test_data["data"]

target_decision = "MTG-PL-2024-01-18-DEC-163491"

# 2. Search through the list
for item in items_list:
    if isinstance(item, dict) and item.get("id", "").endswith(target_decision):
        print("FOUND THE VOTE RESULTS:")
        print(json.dumps(item, indent=4))
        break # This stops the loop and skips the 'else' block below
else:
    # This ONLY runs if the loop finishes without ever hitting 'break'
    print(f"Could not find {target_decision}.")

Could not find MTG-PL-2024-01-18-DEC-163491.


In [2]:
#Since the older votes (2014/2016) did not have outcomes and votes available digitally
#we try this way to get their results
#try with one url first

doc_url = "https://data.europarl.europa.eu/eli/dl/doc/PV-8-2016-04-28-VOT"

# We strictly enforce JSON-LD so we don't get XML again!
headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en"
}

print(f"Fetching document metadata for: {doc_url}\n")

try:
    response = requests.get(doc_url, headers=headers, timeout=15)
    
    if response.status_code == 200:
        doc_data = response.json()
        
        print("=== DOCUMENT METADATA JSON ===")
        print(json.dumps(doc_data, indent=4))
        
    else:
        print(f"Failed to fetch document: HTTP {response.status_code}")
        print(response.text)
        
except Exception as e:
    print(f"Error fetching document: {e}")
                

Fetching document metadata for: https://data.europarl.europa.eu/eli/dl/doc/PV-8-2016-04-28-VOT

=== DOCUMENT METADATA JSON ===
{
    "data": [
        {
            "id": "eli/dl/doc/PV-8-2016-04-28-VOT",
            "type": "Work",
            "parliamentary_term": "org/ep-8",
            "document_date": "2016-04-28",
            "is_annex_of": [
                "eli/dl/doc/PV-8-2016-04-28"
            ],
            "is_realized_by": [
                {
                    "id": "eli/dl/doc/PV-8-2016-04-28-VOT/fr",
                    "type": "Expression",
                    "is_embodied_by": [
                        {
                            "id": "eli/dl/doc/PV-8-2016-04-28-VOT/fr/docx",
                            "type": "Manifestation",
                            "is_exemplified_by": "distribution/doc/PV-8-2016-04-28-VOT-FNL_fr.docx",
                            "media_type": "https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.word